# Face Recognition System using PCA (Eigenfaces) and ANN

This notebook demonstrates a complete end-to-end Face Recognition pipeline. It reduces high-dimensional image matrices into a dense set of features (Principal Components / Eigenfaces) and then classifies them using a Multi-Layer Perceptron (ANN).

## 1. Import Dependencies

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from scipy.spatial.distance import euclidean

## 2. Load and Preprocess the Dataset
Images are read in grayscale, resized to 100x100, and flattened into a 1D vector.

In [ ]:
dataset_path = "dataset"
images = []
labels = []
label_dict = {}

if os.path.exists(dataset_path):
    for label_id, person in enumerate(sorted(os.listdir(dataset_path))):
        person_path = os.path.join(dataset_path, person)
        if not os.path.isdir(person_path): continue
        
        label_dict[label_id] = person
        for img_name in os.listdir(person_path):
            img_path = os.path.join(person_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (100, 100))
                images.append(img.flatten())
                labels.append(label_id)

X = np.array(images)
y = np.array(labels)
print("Dataset Shape:", X.shape)
print("Labels Mapping:", label_dict)

## 3. Train-Test Split
We split the dataset 60% for training and 40% for testing.

In [ ]:
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
    print(f"Training samples: {len(X_train)} \nTesting samples: {len(X_test)}")
else:
    print("Dataset is empty. Please capture images first.")

## 4. PCA - Compute Mean Face and Eigenfaces
Compute the surrogate covariance matrix and perform Eigen Decomposition.

In [ ]:
if len(X) > 0:
    mean_face = np.mean(X_train, axis=0)
    X_train_centered = X_train - mean_face
    X_test_centered = X_test - mean_face
    
    # Surrogate Covariance
    cov_matrix = np.dot(X_train_centered, X_train_centered.T)
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    
    # Sort in descending order
    idx = np.argsort(-eigenvalues)
    eigenvectors = eigenvectors[:, idx]
    
    # Compute actual Eigenfaces
    eigenfaces = np.dot(X_train_centered.T, eigenvectors)
    for i in range(eigenfaces.shape[1]):
        eigenfaces[:, i] /= np.linalg.norm(eigenfaces[:, i])
        
    print("Eigenfaces computed.")

## 5. Visualizing the First 10 Eigenfaces
What do the principal components look like?

In [ ]:
if len(X) > 0:
    plt.figure(figsize=(15, 6))
    for i in range(min(10, eigenfaces.shape[1])):
        plt.subplot(2, 5, i+1)
        # Reshape the flattened array back to 100x100 image
        img = eigenfaces[:, i].reshape(100, 100)
        plt.imshow(img, cmap='gray')
        plt.title(f'Eigenface {i+1}')
        plt.axis('off')
    plt.suptitle('Top 10 Eigenfaces', fontsize=16)
    plt.show()

## 6. Training ANN over Different $k$ values
We test how the number of Eigenfaces affects accuracy.

In [ ]:
def train_evaluate(k):
    ek = eigenfaces[:, :k]
    
    # Project into sub-space
    X_train_proj = np.dot(X_train_centered, ek)
    X_test_proj = np.dot(X_test_centered, ek)
    
    # Train ANN
    model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    model.fit(X_train_proj, y_train)
    
    # Predict
    acc = accuracy_score(y_test, model.predict(X_test_proj))
    return acc

if len(X) > 0:
    k_vals = list(range(10, min(80, eigenfaces.shape[1] + 1), 10))
    accuracies = []
    print("Evaluating Accuracy vs Number of Eigenfaces (k)...")
    for k in k_vals:
        acc = train_evaluate(k)
        accuracies.append(acc)
        print(f"k = {k:02d} | Accuracy = {acc:.2f}")

## 7. Plotting Accuracy vs k

In [ ]:
if len(X) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(k_vals, accuracies, marker='o', linestyle='-', color='b')
    plt.title("Accuracy vs Number of Eigenfaces (k)", fontsize=14)
    plt.xlabel("k (Eigenfaces)", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.grid(True)
    plt.show()

## 8. Imposter Detection & Recognition
Function to detect an individual or flag as Imposter.

In [ ]:
def recognize_or_imposter(image_path, k=40, threshold=5000):
    if not os.path.exists(image_path):
        return f"File {image_path} not found."
        
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img_centered = cv2.resize(img, (100, 100)).flatten() - mean_face
    
    ek = eigenfaces[:, :k]
    img_proj = np.dot(img_centered, ek)
    
    # We need the training projections to compare distances
    X_train_proj = np.dot(X_train_centered, ek)
    distances = [euclidean(img_proj, t) for t in X_train_proj]
    
    if min(distances) > threshold:
        return "Result: Unknown Person (Imposter Detected!)"
        
    # Train final model for prediction (in practice cache this)
    model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    model.fit(X_train_proj, y_train)
    
    pred = model.predict([img_proj])[0]
    return f"Result: Recognized as {label_dict[pred]}"

## Try it out!
Uncomment the cell below and run it with an image from outside your dataset to test imposter detection, or inside your dataset to test recognition.

In [ ]:
# if os.path.exists('test.jpg'):
#     print(recognize_or_imposter('test.jpg'))
# else:
#     print("Add 'test.jpg' to the folder to test recognition.")